# Segment the landscape — a U-Net on Sentinel-2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwinkelmann/GEE_Animation/blob/main/notebooks/08_unet_segmentation.ipynb)

Notebook 07's random forest classifies every pixel **alone**: six reflectance
values in, one class out, neighbours invisible. But land cover has *shape* —
field edges are straight, lakes are connected, settlements clump. A
**convolutional network** sees patches instead of pixels and can learn exactly
that. We use a **U-Net** from
[`segmentation_models_pytorch`](https://github.com/qubvel-org/segmentation_models.pytorch)
(`smp`), the standard library of ready-made segmentation architectures, and the
same Dynamic World labels as notebook 07 — so the two methods are directly
comparable.

One honest expectation up front: we train on a **single scene**, so the network
will learn *this landscape*, not the world. That is deliberate — it keeps the
notebook runnable in minutes — and section 8 spells out what it costs.
A GPU helps but is not required (Colab: *Runtime → Change runtime type → GPU*).

## 1 · Install

In [ ]:
import sys
IS_COLAB = "google.colab" in sys.modules
BRANCH = "main"
URL = f"gee_animation[shapefile] @ git+https://github.com/cwinkelmann/GEE_Animation.git@{BRANCH}"
if IS_COLAB:
    !pip install -q "{URL}"                              # deps + package (first run)
    !pip install -q --force-reinstall --no-deps "{URL}"  # branch moved? force-refresh our code
try:                       # Colab ships torch; a local kernel may lack torch AND smp
    import segmentation_models_pytorch                   # noqa: F401
except ModuleNotFoundError:
    %pip install -q segmentation-models-pytorch
    print("installed into this kernel — if the imports below still fail, restart the kernel and re-run")

## 2 · Authenticate

In [ ]:
import os
PROJECT = os.environ.get("EE_PROJECT", "")   # <-- or paste your own project id here
assert PROJECT, "Set PROJECT to your own Earth Engine Cloud project id — 00_setup.ipynb shows how to create one"

from gee_animation import auth
auth.init(PROJECT)

## 3 · The scene — Grumsin and its surroundings

Same frame as notebooks 06/07: the Grumsin footprint plus a 3 km collar, so
forest, farmland, lakes and settlements are all present.

In [ ]:
from pathlib import Path
import ee
from gee_animation import aoi

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docs" / "aoi").is_dir()), None)
if REPO is not None:
    REGION = str(REPO / "docs" / "aoi" / "wne" / "wne.geojson")
else:   # Colab: fetch the same file
    import urllib.request
    RAW = ("https://raw.githubusercontent.com/cwinkelmann/GEE_Animation/main"
           "/docs/aoi/wne/wne.geojson")
    urllib.request.urlretrieve(RAW, "wne.geojson")
    REGION = "wne.geojson"

START, END = "2025-06-01", "2025-09-01"     # one summer, end exclusive

region_geom = aoi.parse({"geojson": REGION})
frame_bbox = list(aoi.frame_bbox_from_region(region_geom, 3000.0))
frame_geom = aoi.parse({"bbox": frame_bbox})
print("frame bbox:", [round(v, 4) for v in frame_bbox], "(footprint + 3 km)")

## 4 · Download the scene as arrays

A network trains on tensors, not on Earth Engine images, so this time the
composite comes **down**: the same cloud-masked summer median as 06/07, plus the
Dynamic World labels, both as NumPy arrays at 20 m in the frame's UTM zone.
Earth Engine's `NPY` download format makes that a two-request affair — no
GeoTIFF handling needed.

In [ ]:
import io
import urllib.request
import numpy as np
from gee_animation import products

sensor = products.SENSORS["sentinel2"]
BANDS = ["blue", "green", "red", "nir", "swir1", "swir2"]
SCALE, CRS = 20, "EPSG:32633"               # UTM 33N — Brandenburg

coll = (sensor.collection(ee)
        .filterDate(START, END)
        .filterBounds(frame_geom)
        .filter(ee.Filter.lte(sensor.scene_cloud_property, 70)))
coll = sensor.attach_aux(coll, START, END, frame_geom, ee_module=ee)
coll = (coll.map(lambda img: sensor.mask_clouds(img, ee))
            .map(lambda img: sensor.reflectance(img, ee)))
composite = coll.median().clip(frame_geom)

dw = (ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
      .filterDate(START, END).filterBounds(frame_geom)
      .select("label").mode())

def fetch_npy(image, bands):
    url = image.getDownloadURL({"format": "NPY", "bands": bands,
                                "region": frame_geom, "scale": SCALE, "crs": CRS})
    arr = np.load(io.BytesIO(urllib.request.urlopen(url).read()))
    return np.stack([arr[b].astype("float32") for b in bands])

X = fetch_npy(composite, BANDS)                       # (6, H, W) reflectance
y = fetch_npy(dw, ["label"])[0].astype("int64")       # (H, W) class 0..8
print("image", X.shape, "labels", y.shape)

## 5 · Patches, and an honest split

64 × 64-pixel tiles (1.3 km on the ground) cut from the scene, a fifth of them
held out for validation **before** training. Pixels the cloud mask left empty
get the ignore label 255, so they count for nothing — neither in the loss nor in
the accuracy.

In [ ]:
P = 64
STRIDE = 16  # Smaller stride creates overlapping tiles, increasing dataset size
IGNORE = 255


nodata = ~np.isfinite(X).all(axis=0)
X = np.nan_to_num(X, nan=0.0)
y[nodata] = IGNORE

H, W = y.shape
tiles = [(i, j) for i in range(0, H - P + 1, P) for j in range(0, W - P + 1, P)]
rng = np.random.default_rng(1)
rng.shuffle(tiles)
n_val = max(1, len(tiles) // 5)
val_tiles, train_tiles = tiles[:n_val], tiles[n_val:]

def stack(tt):
    xs = np.stack([X[:, i:i+P, j:j+P] for i, j in tt])
    ys = np.stack([y[i:i+P, j:j+P] for i, j in tt])
    return xs, ys

Xtr, ytr = stack(train_tiles)
Xva, yva = stack(val_tiles)
print(f"{len(train_tiles)} training tiles, {len(val_tiles)} validation tiles, "
      f"{(y == IGNORE).mean():.1%} ignored pixels")

## 6 · The U-Net

`smp.Unet` assembles the whole architecture in one call: a ResNet-18 encoder
(ImageNet-pretrained — `smp` adapts the first convolution from 3 to our 6
channels), a decoder that upsamples back to full resolution, and a 9-class
output head. Training is deliberately plain: Adam, cross-entropy that skips the
ignore label, a few epochs.

In [ ]:
import torch
import segmentation_models_pytorch as smp

device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(1)

model = smp.Unet("resnet18", encoder_weights="imagenet",
                 in_channels=len(BANDS), classes=9).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss(ignore_index=IGNORE)

xtr = torch.tensor(Xtr, device=device); ytr_t = torch.tensor(ytr, device=device)
xva = torch.tensor(Xva, device=device); yva_t = torch.tensor(yva, device=device)

EPOCHS, BATCH = 10, 8  # make it 30 epochs if you have time to spare
for epoch in range(1, EPOCHS + 1):
    model.train()
    order = torch.randperm(len(xtr))
    tot = 0.0
    for k in range(0, len(order), BATCH):
        idx = order[k:k + BATCH]
        opt.zero_grad()
        loss = loss_fn(model(xtr[idx]), ytr_t[idx])
        loss.backward(); opt.step()
        tot += float(loss.detach()) * len(idx)
    model.eval()
    with torch.no_grad():
        va_pred = model(xva).argmax(1)
        ok = yva_t != IGNORE
        va_acc = float((va_pred[ok] == yva_t[ok]).float().mean())
    print(f"epoch {epoch:>2}  train loss {tot / len(xtr):.3f}  val pixel accuracy {va_acc:.2f}")

## 7 · Segment the whole scene

One forward pass over the full frame (padded to the encoder's stride), then the
three maps side by side: what the network sees, what it says, and what the
labeler said.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

DW_CLASSES = ["water", "trees", "grass", "flooded_vegetation", "crops",
              "shrub_and_scrub", "built", "bare", "snow_and_ice"]
DW_PALETTE = ["#419bdf", "#397d49", "#88b053", "#7a87c6", "#e49635",
              "#dfc35a", "#c4281b", "#a59b8f", "#b39fe1"]
cmap = ListedColormap(DW_PALETTE)

pad_h, pad_w = (32 - H % 32) % 32, (32 - W % 32) % 32
x_full = torch.tensor(np.pad(X, ((0, 0), (0, pad_h), (0, pad_w))), device=device)
model.eval()
with torch.no_grad():
    pred = model(x_full[None]).argmax(1)[0, :H, :W].cpu().numpy()
pred_masked = np.where(y == IGNORE, np.nan, pred)

rgb = np.clip(np.stack([X[2], X[1], X[0]], -1) / 0.17, 0, 1)
fig, axes = plt.subplots(1, 3, figsize=(19, 6))
for ax, (title, im) in zip(axes, [("summer composite (true colour)", rgb),
                                  ("U-Net segmentation", pred_masked),
                                  ("Dynamic World (the labels)", np.where(y == IGNORE, np.nan, y))]):
    ax.imshow(im, cmap=None if im.ndim == 3 else cmap, vmin=0, vmax=8,
              interpolation="nearest")
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
fig.legend(handles=[plt.Rectangle((0, 0), 1, 1, fc=c) for c in DW_PALETTE],
           labels=DW_CLASSES, loc="lower center", ncol=5, frameon=False)
plt.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()

## 8 · What just happened — and what did not

Look at the middle map next to notebook 07's: the U-Net's fields are **coherent
shapes** with clean edges, where the random forest produced per-pixel salt and
pepper. That is the context the convolutions buy.

Now the honesty, which cuts deeper here than in 07:

- **The validation tiles come from the same scene.** The network has seen this
  landscape's style in every training tile; the score says nothing about the
  next landscape. For a real model you would train on many frames from many
  regions and validate on *regions held out entirely*.
- **The labels are still Dynamic World** — a model's output. We trained a model
  to imitate a model. Both could be wrong about the same pixel, and our
  accuracy would never notice.
- **Pretraining smuggles in knowledge**: the ResNet encoder arrives knowing
  ImageNet's edges and textures. On 50-odd tiles, that is most of why training
  converges at all.

| | 07 — random forest | 08 — U-Net |
|---|---|---|
| sees | one pixel (6 numbers) | a 64 × 64 neighbourhood |
| learns | thresholds on bands | textures, edges, shapes |
| where | Earth Engine servers | your GPU/CPU, on downloaded tensors |
| data appetite | hundreds of points | thousands of labeled tiles (we cheated with one scene) |
| failure smell | salt-and-pepper speckle | confident, smooth — and wrong in new regions |

## Make it yours

- Train longer, or swap `resnet18` for `efficientnet-b0` — one argument.
- Set `encoder_weights=None` and watch what ImageNet was contributing.
- Download a **second frame** (another AOI) and evaluate there: the honest
  number. The gap between the two accuracies is the overfitting you bought.
- Replace Dynamic World with your own polygons rasterised to a label band —
  then the network imitates *you*, which is the point of the craft.

### Questions:
- What is needed to produce a classifier which generalizes well to new regions and can reproduce Google Dynamic World?
- Can you train a model which learns CORINE Landcover classes